# TPE – Fundamentos de la Ciencia de Datos 2026
## EPA Fuel Economy – Model Year 2025 (Test Car List)

Secciones: 1. Análisis exploratorio · 2. Limpieza · 3. Hipótesis propias · 4. Validación de hipótesis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")

# 1. Análisis exploratorio inicial

## 1.1 Origen de los datos

> **Completar con investigación online** (fueleconomy.gov / EPA "Test Car List"):
> - ¿Quién genera los datos y para qué? (EPA + fabricantes; certificación y etiquetas de consumo)
> - ¿Qué es un ensayo FTP, HWY, US06, SC03, Cold CO, CD (Charge Depleting)?
> - ¿Qué significa que una fila sea un *ensayo* y no un vehículo?
> - Fuente/URL y fecha de descarga (el archivo es `25-testcar-2026-01-21`).

## 1.2 Carga de datos

In [ ]:
# El archivo viene con COMA decimal (p. ej. "4,000"). Sin decimal="," pandas lee esas columnas como texto.
df_raw = pd.read_csv("data/epa_testcar_2025.csv", decimal=",", low_memory=False)
df_raw.shape

In [ ]:
df_raw.head()

In [ ]:
df_raw.info()

## 1.3 Caracterización de atributos

Tipo computacional (pandas) vs. tipo estadístico (nominal, ordinal, continuo, discreto). Las columnas numéricas que representan **códigos** (`*Cd`, `*Code`) son categóricas aunque sean enteros.

In [ ]:
info = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "nulos": df_raw.isna().sum(),
    "% nulos": (df_raw.isna().mean() * 100).round(1),
    "únicos": df_raw.nunique(),
})
info.sort_values("% nulos", ascending=False)

> **Completar:** tabla propia con columna *tipo estadístico* y *qué representa* para las variables que vamos a usar:
> `Vehicle Type`, `Test Veh Displacement (L)`, `Rated Horsepower`, `# of Cylinders and Rotors`, `Tested Transmission Type`, `# of Gears`, `Drive System Description`, `Equivalent Test Weight (lbs.)`, `Test Category`, `Test Fuel Type Description`, `RND_ADJ_FE`.

| Variable | Tipo estadístico | Qué representa |
|---|---|---|
| ... | ... | ... |

## 1.4 Granularidad: una fila = un ensayo

In [ ]:
clave = ["Veh Mfr Code", "Test Vehicle ID", "Test Veh Configuration #"]
print("Filas:", len(df_raw))
print("Configuraciones únicas (clave EPA):", df_raw.groupby(clave).ngroups)
print("Filas completamente duplicadas:", df_raw.duplicated().sum())
print("Test Number repetidos:", df_raw["Test Number"].duplicated().sum())
print()
print("Ensayos por configuración:")
print(df_raw.groupby(clave).size().value_counts().sort_index())
print()
print(df_raw["Test Category"].value_counts())

> **Completar:** ¿por qué hay filas con el mismo `Test Number`? ¿Qué implica para la independencia de las observaciones en los tests de la sección 4? (se resuelve en la limpieza).

## 1.5 Variables categóricas

In [ ]:
cats = ["Vehicle Type", "Tested Transmission Type", "Drive System Description",
        "Test Category", "Test Fuel Type Description", "Aftertreatment Device Desc",
        "Test Originator", "Analytically Derived FE?"]
for c in cats:
    print(f"\n--- {c} ---")
    print(df_raw[c].value_counts(dropna=False))

## 1.6 Valores nulos

In [ ]:
nulos = (df_raw.isna().mean() * 100).round(1)
nulos = nulos[nulos > 0].sort_values()
fig, ax = plt.subplots(figsize=(8, 9))
nulos.plot.barh(ax=ax, color="steelblue")
ax.set_xlabel("% de nulos")
ax.set_title("Porcentaje de nulos por columna")
plt.tight_layout(); plt.show()

In [ ]:
# ¿De dónde vienen los nulos de cilindros? ¿Y los de RND_ADJ_FE?
print(df_raw.loc[df_raw["# of Cylinders and Rotors"].isna(), "Test Fuel Type Description"].value_counts())
print()
print(df_raw.loc[df_raw["RND_ADJ_FE"].isna(), "Test Category"].value_counts())

> **Completar:** ¿qué mecanismo de faltantes (MCAR/MAR/MNAR) sugiere cada grupo de nulos? Ej.: cilindros nulos ⇒ vehículos eléctricos/hidrógeno (el atributo no aplica); ADFE ~95 % nulos ⇒ solo existe cuando `Analytically Derived FE? = Yes`; `FE Bag 3/4`, `PM`, `Averaging *` ⇒ depende del tipo de ensayo.

## 1.7 Resumen estadístico

In [ ]:
num_cols = ["Test Veh Displacement (L)", "Rated Horsepower", "# of Cylinders and Rotors", "# of Gears",
            "Equivalent Test Weight (lbs.)", "Axle Ratio", "N/V Ratio", "CO2 (g/mi)", "RND_ADJ_FE"]
df_raw[num_cols].describe().T.round(2)

### Valores centinela y atípicos sospechosos
El `describe()` muestra cosas que no pueden ser reales (p. ej. `RND_ADJ_FE` máx. = 10000, cilindrada máx. = 100 L, `N/V Ratio` máx. = 999). Los inspeccionamos.

In [ ]:
print("RND_ADJ_FE >= 999 :", (df_raw["RND_ADJ_FE"] >= 999).sum(), "filas ->", df_raw.loc[df_raw["RND_ADJ_FE"] >= 999, "RND_ADJ_FE"].unique())
print("Cilindrada >= 20 L :", (df_raw["Test Veh Displacement (L)"] >= 20).sum(), "filas ->", df_raw.loc[df_raw["Test Veh Displacement (L)"] >= 20, "Test Fuel Type Description"].unique())
print("N/V Ratio == 999   :", (df_raw["N/V Ratio"] == 999).sum())
print("Axle Ratio == 9.99 :", (df_raw["Axle Ratio"] >= 9.99).sum(), "| Axle Ratio == 0:", (df_raw["Axle Ratio"] == 0).sum())
print("Potencia <= 5 HP   :", (df_raw["Rated Horsepower"] <= 5).sum())
print("# of Gears == 1    :", (df_raw["# of Gears"] == 1).sum(), "(CVT / eléctricos: no es una caja de 1 marcha)")
df_raw.loc[df_raw["RND_ADJ_FE"] >= 999, ["Represented Test Veh Make", "Test Category", "Test Fuel Type Description", "RND_ADJ_FE"]].head(10)

> **Completar:** 10000/9999.99…/999 y 99.999 son **códigos de "dato no disponible"**, no mediciones. Si no se tratan, arruinan media, desvío, asimetría y cualquier test. Se corrigen en la sección 2.

## 1.8 Distribuciones y outliers
Para que los gráficos reflejen vehículos reales, en esta sección grafico excluyendo (solo para visualizar) los eléctricos/hidrógeno y los centinelas. La limpieza formal y su justificación están en la sección 2.

In [ ]:
comb_vis = df_raw[
    ~df_raw["Test Fuel Type Description"].isin(["Electricity", "Hydrogen 5"])
    & (df_raw["RND_ADJ_FE"] < 999)
]
vars_vis = ["RND_ADJ_FE", "Rated Horsepower", "Equivalent Test Weight (lbs.)", "Test Veh Displacement (L)"]

fig, axes = plt.subplots(2, 4, figsize=(18, 7))
for i, v in enumerate(vars_vis):
    sns.histplot(comb_vis[v].dropna(), kde=True, ax=axes[0, i])
    axes[0, i].set_title(v)
    sns.boxplot(x=comb_vis[v].dropna(), ax=axes[1, i])
plt.tight_layout(); plt.show()

In [ ]:
res = pd.DataFrame({
    "asimetría": comb_vis[vars_vis].apply(lambda s: stats.skew(s.dropna())),
    "curtosis (exceso)": comb_vis[vars_vis].apply(lambda s: stats.kurtosis(s.dropna())),
}).round(2)
res

In [ ]:
def n_outliers_iqr(s):
    s = s.dropna(); q1, q3 = s.quantile([.25, .75]); i = q3 - q1
    return int(((s < q1 - 1.5 * i) | (s > q3 + 1.5 * i)).sum())
pd.Series({v: n_outliers_iqr(comb_vis[v]) for v in vars_vis}, name="outliers (regla 1.5·IQR)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
sns.countplot(data=comb_vis, y="Drive System Description", ax=axes[0])
sns.countplot(data=comb_vis, y="Tested Transmission Type", ax=axes[1])
sns.countplot(data=comb_vis, x="# of Cylinders and Rotors", ax=axes[2])
plt.tight_layout(); plt.show()

> **Completar (interpretación):** forma de cada distribución (simétrica/sesgada, uni/multimodal), outliers y si son errores o autos reales (superdeportivos de >1000 HP), qué dice el sesgo sobre qué estadístico de resumen conviene (media vs. mediana).

## 1.9 Relaciones preliminares y ProfileReport

In [ ]:
cols_corr = ["RND_ADJ_FE", "Rated Horsepower", "Equivalent Test Weight (lbs.)", "Test Veh Displacement (L)", "# of Cylinders and Rotors"]
sns.heatmap(comb_vis[cols_corr].corr(method="spearman"), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlación de Spearman (combustión, sin centinelas)"); plt.show()

In [ ]:
# Reporte automático de apoyo (se guarda en un archivo; está en .gitignore si no se quiere entregar)
from ydata_profiling import ProfileReport
import os; os.makedirs("reports", exist_ok=True)
ProfileReport(comb_vis[vars_vis + ["Vehicle Type", "Drive System Description", "Tested Transmission Type"]],
              title="EPA 2025 – exploración inicial", minimal=True).to_file("reports/profile_eda.html")